In [ ]:
!pip install opendatasets


In [ ]:
import opendatasets as od

od.download(
    "https://www.kaggle.com/datasets/saeedazfar/customized-cotton-disease-dataset"
)

Skipping, found downloaded files in "./customized-cotton-disease-dataset" (use force=True to force download)
unzip:  cannot find or open customized-cotton-disease-dataset.zip, customized-cotton-disease-dataset.zip.zip or customized-cotton-disease-dataset.zip.ZIP.


In [ ]:
!ls


customized-cotton-disease-dataset  sample_data


In [ ]:
!find ./customized-cotton-disease-dataset -type d

./customized-cotton-disease-dataset
./customized-cotton-disease-dataset/Customized Cotton Dataset-Complete
./customized-cotton-disease-dataset/Customized Cotton Dataset-Complete/content
./customized-cotton-disease-dataset/Customized Cotton Dataset-Complete/content/trainning
./customized-cotton-disease-dataset/Customized Cotton Dataset-Complete/content/trainning/Cotton leaves - Training
./customized-cotton-disease-dataset/Customized Cotton Dataset-Complete/content/trainning/Cotton leaves - Training/800 Images
./customized-cotton-disease-dataset/Customized Cotton Dataset-Complete/content/trainning/Cotton leaves - Training/800 Images/Cotton Boll Rot
./customized-cotton-disease-dataset/Customized Cotton Dataset-Complete/content/trainning/Cotton leaves - Training/800 Images/Bacterial blight
./customized-cotton-disease-dataset/Customized Cotton Dataset-Complete/content/trainning/Cotton leaves - Training/800 Images/Powdery mildew
./customized-cotton-disease-dataset/Customized Cotton Dataset-C

In [ ]:
train_path = "/content/customized-cotton-disease-dataset/Cotton-Disease-Training/trainning/Cotton leaves - Training/800 Images"

val_path = "/content/customized-cotton-disease-dataset/Cotton-Disease-Validation/validation/Cotton plant disease-Validation/Cotton plant disease-Validation"

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(224, 224),
    batch_size=32
)

Found 6628 files belonging to 8 classes.


In [ ]:
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    val_path,
    image_size=(224, 224),
    batch_size=32
)

Found 357 files belonging to 8 classes.


In [ ]:
class_names = train_dataset.class_names

print(class_names)

['Aphids', 'Army worm', 'Bacterial blight', 'Cotton Boll Rot', 'Green Cotton Boll', 'Healthy', 'Powdery mildew', 'Target spot']


In [ ]:
normalization_layer = layers.Rescaling(1./255)

train_dataset = train_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)

validation_dataset = validation_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
base_model.trainable = False

In [ ]:
model = tf.keras.Sequential([
    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.3),

    layers.Dense(128, activation='relu'),

    layers.Dense(len(class_names), activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10
)

Epoch 1/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 546s 3s/step - accuracy: 0.1299 - loss: 2.0965 - val_accuracy: 0.1709 - val_loss: 2.0711
Epoch 2/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 567s 3s/step - accuracy: 0.1471 - loss: 2.0768 - val_accuracy: 0.1709 - val_loss: 2.0617
Epoch 3/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 563s 3s/step - accuracy: 0.1489 - loss: 2.0721 - val_accuracy: 0.1709 - val_loss: 2.0601
Epoch 4/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 546s 3s/step - accuracy: 0.1459 - loss: 2.0699 - val_accuracy: 0.1709 - val_loss: 2.0595
Epoch 5/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 540s 3s/step - accuracy: 0.1599 - loss: 2.0621 - val_accuracy: 0.1709 - val_loss: 2.0569
Epoch 6/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 538s 3s/step - accuracy: 0.1705 - loss: 2.0563 - val_accuracy: 0.1709 - val_loss: 2.0546
Epoch 7/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 554s 3s/step - accuracy: 0.1646 - loss: 2.0590 - val_accuracy: 0.1709 - val_loss: 2.0538
Epoch 8/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 558s 3s/step - accuracy: 0.1619 - loss: 2.0529 - val_accu

In [ ]:
model.save("cotton_disease_model.h5")